# 03 QC: Combined variant summary figures by run

This notebook creates combined summary figures across Run1, Run2, Run3, and Run4 using the per-run Excel summary files.

Figures are generated separately for each mean target coverage cutoff:
- 1x
- 20x
- 30x

The notebook compares:

- variant class counts by run
- 16-gene panel variant counts by run
- variant class percentages by run
- 16-gene panel percentages by run
- DeepVariant versus Mutect2 mean counts by variant class


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


output_parent = Path(r"C:\Users\katya\Box\KD_SUIP_noncoding_project\sequencing_and_variantcalling_overview")

figure_dir = output_parent / "03_combined_figures"
figure_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
#runs = ["Run1", "Run2", "Run3", "Run4"]
runs = ["Run2"]
run_prefixes = [run.lower() for run in runs]

coverage_cutoffs = [None, 1, 20, 30]

variant_classes = ["SNV", "insertion", "deletion", "indel", "substitution"]

panel_genes = [
    "ATM", "BARD1", "BRCA1", "BRCA2",
    "CDH1", "CDKN2A", "CHEK2", "MLH1",
    "MSH2", "MSH6", "PALB2", "PMS2",
    "PTEN", "RAD51C", "RAD51D", "TP53",
]

## Load per-run summary files

Each coverage cutoff uses the per-run Excel summary files created earlier in the QC workflow.

In [ ]:
def coverage_label(coverage_cutoff):
    return "all_samples" if coverage_cutoff is None else f"mean_cov_gt_{coverage_cutoff}x"


def coverage_title(coverage_cutoff):
    return "All samples" if coverage_cutoff is None else f"Mean target coverage > {coverage_cutoff}x"


def load_run_summaries(coverage_cutoff):
    label = coverage_label(coverage_cutoff)

    sheets = {
        "variant_class": "variant_class_summary",
        "gene": "gene_summary_16",
        "class_pct": "class_by_sample_pct",
        "gene_pct": "gene_by_sample_pct",
    }

    summaries = {key: {} for key in sheets}

    for run, prefix in zip(runs, run_prefixes):
        excel_file = output_parent / prefix / label / f"{prefix}_complete_variant_summary_{label}.xlsx"

        for key, sheet in sheets.items():
            summaries[key][run] = pd.read_excel(excel_file, sheet_name=sheet)

        print(f"Loaded {run}, {label}")

    return summaries


summary_by_cutoff = {
    cutoff: load_run_summaries(cutoff)
    for cutoff in coverage_cutoffs
}

## Plot helper functions

In [ ]:
def combined_mean(df):
    return (df["deepvariant_mean"] + df["mutect2_mean"]) / 2


def save_show(filename):
    plt.tight_layout()
    plt.savefig(figure_dir / filename, dpi=300)
    plt.show()
    plt.close()


def add_grid():
    plt.grid(axis="y", alpha=0.3)
    plt.gca().set_axisbelow(True)


def grouped_bar(plot_df, x_col, y_col, order, title, xlabel, ylabel, filename, figsize):
    x = np.arange(len(order))
    width = 0.8 / len(runs)

    plt.figure(figsize=figsize)

    for i, run in enumerate(runs):
        vals = (
            plot_df[plot_df["run"] == run]
            .set_index(x_col)
            .reindex(order)[y_col]
            .fillna(0)
        )

        plt.bar(x + i * width, vals, width, label=run)

    plt.xticks(x + width * 1.5, order, rotation=45, ha="right")
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    add_grid()
    save_show(filename)

## Variant class mean counts by run

This plot compares the combined mean variant count for each variant class across runs.

In [ ]:
for cutoff in coverage_cutoffs:
    label = coverage_label(cutoff)
    title_label = coverage_title(cutoff)
    variant_class_by_run = summary_by_cutoff[cutoff]["variant_class"]

    rows = []

    for run in runs:
        df = variant_class_by_run[run].copy()
        df["combined_mean"] = combined_mean(df)

        for vc in variant_classes:
            val = df.loc[df["Variant.Class"] == vc, "combined_mean"]
            rows.append({
                "run": run,
                "variant_class": vc,
                "combined_mean": val.iloc[0] if len(val) else 0,
            })

    plot_df = pd.DataFrame(rows)

    grouped_bar(
        plot_df=plot_df,
        x_col="variant_class",
        y_col="combined_mean",
        order=variant_classes,
        title=f"Mean variant class counts by run\n{title_label}",
        xlabel="Variant class",
        ylabel="Mean variants per sample",
        filename=f"ONLY_03_variant_class_combined_mean_by_run_{label}.png",
        figsize=(10, 6),
    )

## 16-gene panel mean counts by run

This plot compares the combined mean variant count for each panel gene across runs.

Genes are ordered from largest to smallest using Run2.

In [ ]:
for cutoff in coverage_cutoffs:
    label = coverage_label(cutoff)
    title_label = coverage_title(cutoff)
    gene_by_run = summary_by_cutoff[cutoff]["gene"]

    run2_df = gene_by_run["Run2"].copy()
    run2_df["combined_mean"] = combined_mean(run2_df)

    gene_order = (
        run2_df
        .set_index("Gene")
        .reindex(panel_genes)
        .sort_values("combined_mean", ascending=False)
        .index
        .tolist()
    )

    rows = []

    for run in runs:
        df = gene_by_run[run].copy()
        df["combined_mean"] = combined_mean(df)

        for gene in gene_order:
            val = df.loc[df["Gene"] == gene, "combined_mean"]
            rows.append({
                "run": run,
                "gene": gene,
                "combined_mean": val.iloc[0] if len(val) else 0,
            })

    plot_df = pd.DataFrame(rows)

    grouped_bar(
        plot_df=plot_df,
        x_col="gene",
        y_col="combined_mean",
        order=gene_order,
        title=f"Mean variants per gene by run\n{title_label}; genes ordered by Run2",
        xlabel="Gene",
        ylabel="Mean variants per sample",
        filename=f"ONLY_03_gene_combined_mean_by_run_{label}.png", #ordered by run 2
        figsize=(14, 6),
    )

## Variant class percentages by run

This plot compares the average percentage of variants in each variant class across runs.

In [ ]:
for cutoff in coverage_cutoffs:
    label = coverage_label(cutoff)
    title_label = coverage_title(cutoff)
    class_pct_by_run = summary_by_cutoff[cutoff]["class_pct"]

    rows = []

    for run in runs:
        summary = (
            class_pct_by_run[run]
            .groupby("Variant.Class")["pct_of_total_variants"]
            .mean()
            .reindex(variant_classes, fill_value=0)
            .reset_index()
        )

        for _, row in summary.iterrows():
            rows.append({
                "run": run,
                "variant_class": row["Variant.Class"],
                "mean_pct": row["pct_of_total_variants"],
            })

    plot_df = pd.DataFrame(rows)

    grouped_bar(
        plot_df=plot_df,
        x_col="variant_class",
        y_col="mean_pct",
        order=variant_classes,
        title=f"Variant class percentages by run\n{title_label}",
        xlabel="Variant class",
        ylabel="Mean % of total variants",
        filename=f"ONLY_03_variant_class_pct_by_run_{label}.png",
        figsize=(10, 6),
    )

## 16-gene panel percentages by run

This plot compares the average percentage of variants in each panel gene across runs.

Genes are ordered from largest to smallest using Run2.

In [ ]:
for cutoff in coverage_cutoffs:
    label = coverage_label(cutoff)
    title_label = coverage_title(cutoff)
    gene_pct_by_run = summary_by_cutoff[cutoff]["gene_pct"]

    gene_order = (
        gene_pct_by_run["Run2"]
        .groupby("Gene")["pct_of_total_variants"]
        .mean()
        .reindex(panel_genes, fill_value=0)
        .sort_values(ascending=False)
        .index
        .tolist()
    )

    rows = []

    for run in runs:
        summary = (
            gene_pct_by_run[run]
            .groupby("Gene")["pct_of_total_variants"]
            .mean()
            .reindex(gene_order, fill_value=0)
            .reset_index()
        )

        for _, row in summary.iterrows():
            rows.append({
                "run": run,
                "gene": row["Gene"],
                "mean_pct": row["pct_of_total_variants"],
            })

    plot_df = pd.DataFrame(rows)

    grouped_bar(
        plot_df=plot_df,
        x_col="gene",
        y_col="mean_pct",
        order=gene_order,
        title=f"Gene variant percentages by run\n{title_label}; genes ordered by Run2",
        xlabel="Gene",
        ylabel="Mean % of total variants",
        filename=f"ONLY_03_gene_pct_by_run_{label}.png", #ordered by run 2
        figsize=(14, 6),
    )

## DeepVariant and Mutect2 variant class counts

This section creates one plot per variant class comparing DeepVariant and Mutect2 mean counts across runs.

In [ ]:
for cutoff in coverage_cutoffs:
    label = coverage_label(cutoff)
    title_label = coverage_title(cutoff)
    variant_class_by_run = summary_by_cutoff[cutoff]["variant_class"]

    for vc in variant_classes:
        rows = []

        for run in runs:
            df = variant_class_by_run[run]
            row = df[df["Variant.Class"] == vc]

            rows.append({
                "run": run,
                "deepvariant_mean": row["deepvariant_mean"].iloc[0] if len(row) else 0,
                "mutect2_mean": row["mutect2_mean"].iloc[0] if len(row) else 0,
            })

        class_df = pd.DataFrame(rows)

        x = np.arange(len(runs))
        width = 0.35

        plt.figure(figsize=(8, 5))
        plt.bar(x - width / 2, class_df["deepvariant_mean"], width, label="DeepVariant")
        plt.bar(x + width / 2, class_df["mutect2_mean"], width, label="Mutect2")

        plt.xticks(x, runs)
        plt.xlabel("Run")
        plt.ylabel("Mean variants per sample")
        plt.title(f"{vc} mean count by caller and run\n{title_label}")
        plt.legend()
        add_grid()
        safe_vc = vc.replace(" ", "_").lower()
        save_show(f"ONLY_03_{safe_vc}_mean_by_caller_and_run_{label}.png")

In [ ]:
for coverage_cutoff in coverage_cutoffs:
    filter_suffix = f"mean_cov_gt_{coverage_cutoff}x"

    variant_class_by_run, gene_by_run, class_pct_by_run, gene_pct_by_run = load_run_summaries(filter_suffix)

    # -----------------------------
    # 1. Variant class mean count graph
    # -----------------------------
    plot_rows = []

    for run in runs:
        df = variant_class_by_run[run].copy()
        df["combined_mean"] = combined_mean(df)

        for vc in variant_classes:
            val = df.loc[df["Variant.Class"] == vc, "combined_mean"]
            plot_rows.append({
                "run": run,
                "variant_class": vc,
                "combined_mean": val.iloc[0] if len(val) > 0 else 0
            })

    variant_class_plot_df = pd.DataFrame(plot_rows)

    x = np.arange(len(variant_classes))
    width = 0.2

    plt.figure(figsize=(10, 6))

    for i, run in enumerate(runs):
        vals = (
            variant_class_plot_df[variant_class_plot_df["run"] == run]
            .set_index("variant_class")
            .loc[variant_classes, "combined_mean"]
        )

        plt.bar(x + i * width, vals, width, label=run)

    plt.xticks(x + width * 1.5, variant_classes, rotation=45, ha="right")
    plt.xlabel("Variant class")
    plt.ylabel("Mean variants per sample")
    plt.title(f"Mean variant class counts by run\nMean target coverage > {coverage_cutoff}x")
    plt.legend()
    add_grid()
    plt.tight_layout()

    out_png = figure_dir / f"ONLY_variant_class_combined_mean_by_run_{filter_suffix}.png"
    plt.savefig(out_png, dpi=300)
    plt.show()

    # -----------------------------
    # 2. Gene mean count graph
    # Order genes largest to smallest by Run2
    # -----------------------------
    run2_gene_df = gene_by_run["Run2"].copy()
    run2_gene_df["combined_mean"] = combined_mean(run2_gene_df)

    gene_order = (
        run2_gene_df
        .set_index("Gene")
        .reindex(panel_genes)
        .sort_values("combined_mean", ascending=False)
        .index
        .tolist()
    )

    plot_rows = []

    for run in runs:
        df = gene_by_run[run].copy()
        df["combined_mean"] = combined_mean(df)

        for gene in gene_order:
            val = df.loc[df["Gene"] == gene, "combined_mean"]
            plot_rows.append({
                "run": run,
                "gene": gene,
                "combined_mean": val.iloc[0] if len(val) > 0 else 0
            })

    gene_plot_df = pd.DataFrame(plot_rows)

    x = np.arange(len(gene_order))
    width = 0.2

    plt.figure(figsize=(14, 6))

    for i, run in enumerate(runs):
        vals = (
            gene_plot_df[gene_plot_df["run"] == run]
            .set_index("gene")
            .loc[gene_order, "combined_mean"]
        )

        plt.bar(x + i * width, vals, width, label=run)

    plt.xticks(x + width * 1.5, gene_order, rotation=45, ha="right")
    plt.xlabel("Gene")
    plt.ylabel("Mean variants per sample")
    plt.title(f"Mean variants per gene by run\nMean target coverage > {coverage_cutoff}x; genes ordered by Run2")
    plt.legend()
    add_grid()
    plt.tight_layout()

    out_png = figure_dir / f"ONLY_gene_combined_mean_by_run_ordered_by_run2_{filter_suffix}.png"
    plt.savefig(out_png, dpi=300)
    plt.show()

    # -----------------------------
    # 3. Variant class percentage graph
    # -----------------------------
    plot_rows = []

    for run in runs:
        df = class_pct_by_run[run].copy()

        summary = (
            df.groupby("Variant.Class")["pct_of_total_variants"]
            .mean()
            .reindex(variant_classes, fill_value=0)
            .reset_index()
        )

        for _, row in summary.iterrows():
            plot_rows.append({
                "run": run,
                "variant_class": row["Variant.Class"],
                "mean_pct": row["pct_of_total_variants"]
            })

    variant_class_pct_plot_df = pd.DataFrame(plot_rows)

    x = np.arange(len(variant_classes))
    width = 0.2

    plt.figure(figsize=(10, 6))

    for i, run in enumerate(runs):
        vals = (
            variant_class_pct_plot_df[variant_class_pct_plot_df["run"] == run]
            .set_index("variant_class")
            .loc[variant_classes, "mean_pct"]
        )

        plt.bar(x + i * width, vals, width, label=run)

    plt.xticks(x + width * 1.5, variant_classes, rotation=45, ha="right")
    plt.xlabel("Variant class")
    plt.ylabel("Mean % of total variants")
    plt.title(f"Variant class percentages by run\nMean target coverage > {coverage_cutoff}x")
    plt.legend()
    add_grid()
    plt.tight_layout()

    out_png = figure_dir / f"ONLY_variant_class_pct_by_run_{filter_suffix}.png"
    plt.savefig(out_png, dpi=300)
    plt.show()

    # -----------------------------
    # 4. Gene percentage graph
    # Order genes largest to smallest by Run2 %
    # -----------------------------
    run2_gene_pct = (
        gene_pct_by_run["Run2"]
        .groupby("Gene")["pct_of_total_variants"]
        .mean()
        .reindex(panel_genes, fill_value=0)
        .sort_values(ascending=False)
    )

    gene_pct_order = run2_gene_pct.index.tolist()

    plot_rows = []

    for run in runs:
        df = gene_pct_by_run[run].copy()

        summary = (
            df.groupby("Gene")["pct_of_total_variants"]
            .mean()
            .reindex(gene_pct_order, fill_value=0)
            .reset_index()
        )

        for _, row in summary.iterrows():
            plot_rows.append({
                "run": run,
                "gene": row["Gene"],
                "mean_pct": row["pct_of_total_variants"]
            })

    gene_pct_plot_df = pd.DataFrame(plot_rows)

    x = np.arange(len(gene_pct_order))
    width = 0.2

    plt.figure(figsize=(14, 6))

    for i, run in enumerate(runs):
        vals = (
            gene_pct_plot_df[gene_pct_plot_df["run"] == run]
            .set_index("gene")
            .loc[gene_pct_order, "mean_pct"]
        )

        plt.bar(x + i * width, vals, width, label=run)

    plt.xticks(x + width * 1.5, gene_pct_order, rotation=45, ha="right")
    plt.xlabel("Gene")
    plt.ylabel("Mean % of total variants")
    plt.title(f"Gene variant percentages by run\nMean target coverage > {coverage_cutoff}x; genes ordered by Run2")
    plt.legend()
    add_grid()
    plt.tight_layout()

    out_png = figure_dir / f"ONLY_gene_pct_by_run_ordered_by_run2_{filter_suffix}.png"
    plt.savefig(out_png, dpi=300)
    plt.show()

    # -----------------------------
    # 5. Separate variant class graphs by caller
    # SNV, insertion, deletion, indel, substitution
    # -----------------------------
    for vc in variant_classes:
        rows = []

        for run in runs:
            df = variant_class_by_run[run].copy()
            row = df[df["Variant.Class"] == vc]

            if len(row) == 0:
                dv_mean = 0
                mt_mean = 0
            else:
                dv_mean = row["deepvariant_mean"].iloc[0]
                mt_mean = row["mutect2_mean"].iloc[0]

            rows.append({
                "run": run,
                "deepvariant_mean": dv_mean,
                "mutect2_mean": mt_mean
            })

        class_df = pd.DataFrame(rows)

        x = np.arange(len(runs))
        width = 0.35

        plt.figure(figsize=(8, 5))
        plt.bar(x - width / 2, class_df["deepvariant_mean"], width, label="DeepVariant")
        plt.bar(x + width / 2, class_df["mutect2_mean"], width, label="Mutect2")

        plt.xticks(x, runs)
        plt.xlabel("Run")
        plt.ylabel("Mean variants per sample")
        plt.title(f"{vc} mean count by caller and run\nMean target coverage > {coverage_cutoff}x")
        plt.legend()
        add_grid()
        plt.tight_layout()

        safe_vc = vc.replace(" ", "_").lower()
        out_png = figure_dir / f"ONLY_{safe_vc}_mean_by_caller_and_run_{filter_suffix}.png"
        plt.savefig(out_png, dpi=300)
        plt.show()